In [5]:
# stage14-Deployment

import os, json, time, warnings, importlib.util, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


REPO = Path.cwd()
for _ in range(8):
    if (REPO / "homework").exists() or (REPO / ".git").exists():
        break
    REPO = REPO.parent

H14   = REPO / "homework" / "homework14"
H13   = REPO / "homework" / "homework13"
DATA  = H14 / "data"
SRC   = H14 / "src"
NB    = H14 / "notebooks"
DOCS  = H14 / "docs"
REPS  = H14 / "reports"
MODEL = H14 / "model"

for d in [DATA/"raw", DATA/"processed", SRC, NB, DOCS, REPS, MODEL]:
    d.mkdir(parents=True, exist_ok=True)


from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import joblib

src_model = H13 / "model" / "model.pkl"
src_meta  = H13 / "model" / "metadata.json"

if src_model.exists():
    joblib.dump(joblib.load(src_model), MODEL / "model.pkl")  # copy
    (MODEL / "metadata.json").write_text((src_meta.read_text() if src_meta.exists() else "{}"), encoding="utf-8")
else:
    # synthesize a small 2-feature regression to keep endpoints working
    rng = np.random.default_rng(42)
    n = 400
    risk = rng.normal(50, 12, n)
    mkt  = rng.normal(100, 40, n)
    y    = 20000 + 110*risk + 8*mkt + rng.normal(0, 2500, n)
    df = pd.DataFrame({"risk_index_var95": risk, "marketing_spend": mkt, "revenue": y})
    TARGET = "revenue"
    X = df[["risk_index_var95","marketing_spend"]].copy()
    y = df[TARGET].copy()
    Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.2,random_state=14)
    pipe = Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler()),("lr",LinearRegression())]).fit(Xtr,ytr)
    yhat = pipe.predict(Xte)
    rmse = float(np.sqrt(mean_squared_error(yte,yhat)))
    joblib.dump(pipe, MODEL / "model.pkl")
    meta = {"target": TARGET, "feature_names": X.columns.tolist(), "rmse_test": rmse,
            "train_rows": int(len(Xtr)), "test_rows": int(len(Xte))}
    (MODEL / "metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

print("[model] Ready at:", MODEL / "model.pkl")


app_py = '''import time, io, json, logging
from pathlib import Path

import numpy as np
import pandas as pd
from flask import Flask, request, jsonify, send_file, Response
import joblib

# Prometheus metrics
from prometheus_client import Counter, Histogram, Gauge, generate_latest, CONTENT_TYPE_LATEST

HERE = Path(__file__).resolve().parents[1]
MODEL_PATH = HERE / "model" / "model.pkl"
META_PATH  = HERE / "model" / "metadata.json"

# Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("app")

# Load model & metadata
model = joblib.load(MODEL_PATH)
meta  = json.loads(META_PATH.read_text(encoding="utf-8")) if META_PATH.exists() else {}
FEATURES = meta.get("feature_names", [])

# Metrics
REQ_COUNT   = Counter("api_requests_total", "Total API requests", ["endpoint","method","code"])
REQ_LATENCY = Histogram("api_request_latency_seconds", "Request latency", ["endpoint"])
MODEL_READY = Gauge("model_ready", "Model readiness (1 ready, 0 not)")

# App
app = Flask(__name__)
MODEL_READY.set(1.0 if model is not None else 0.0)

def _coerce_df(payload):
    if isinstance(payload, dict):
        df = pd.DataFrame([payload])
    elif isinstance(payload, list):
        df = pd.DataFrame(payload)
    else:
        raise ValueError("Payload must be an object or list of objects")
    # ensure expected columns exist and are ordered; imputer handles NaN
    if not FEATURES:
        raise ValueError("FEATURES list is empty; check metadata.json")
    for c in FEATURES:
        if c not in df.columns:
            df[c] = np.nan
    df = df[FEATURES]
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

@app.before_request
def _start_timer():
    request._start_time = time.perf_counter()

@app.after_request
def _record_metrics(resp):
    try:
        dt = time.perf_counter() - getattr(request, "_start_time", time.perf_counter())
        REQ_LATENCY.labels(request.path).observe(dt)
        REQ_COUNT.labels(request.path, request.method, resp.status_code).inc()
    except Exception:
        pass
    return resp

@app.get("/health")
def health():
    return jsonify(status="ok", model=str(MODEL_PATH.name), features=FEATURES, target=meta.get("target"))

@app.get("/ready")
def ready():
    try:
        # try a dummy predict with NaNs -> imputer path
        if not FEATURES: 
            return jsonify(ready=False, reason="no FEATURES"), 500
        row = {c: np.nan for c in FEATURES}
        df = _coerce_df(row)
        _ = model.predict(df)
        return jsonify(ready=True)
    except Exception as e:
        return jsonify(ready=False, error=str(e)), 500

@app.post("/predict")
def predict_post():
    try:
        payload = request.get_json(force=True, silent=False)
        if isinstance(payload, dict) and "records" in payload:
            df = _coerce_df(payload["records"])
        else:
            df = _coerce_df(payload)
        preds = model.predict(df).tolist()
        return jsonify(predictions=preds, n=len(preds))
    except Exception as e:
        log.exception("predict error")
        return jsonify(error=str(e)), 400

@app.get("/predict/<x1>")
@app.get("/predict/<x1>/<x2>")
def predict_path(x1, x2=None):
    try:
        if not FEATURES:
            return jsonify(error="Model feature list is empty"), 400
        row = {FEATURES[0]: float(x1)}
        if x2 is not None and len(FEATURES) > 1:
            row[FEATURES[1]] = float(x2)
        df = _coerce_df(row)
        pred = float(model.predict(df)[0])
        return jsonify(prediction=pred)
    except Exception as e:
        log.exception("predict_path error")
        return jsonify(error=str(e)), 400

@app.get("/metrics")
def metrics():
    return Response(generate_latest(), mimetype=CONTENT_TYPE_LATEST)

if __name__ == "__main__":
    # Windows-friendly local run (waitress); Dockerfile uses gunicorn
    try:
        from waitress import serve
        serve(app, host="127.0.0.1", port=8000)
    except Exception:
        app.run(host="127.0.0.1", port=8000, debug=False)
'''
(SRC / "app.py").write_text(app_py, encoding="utf-8")
print("[api] Wrote:", SRC / "app.py")


reqs = """flask
pandas
numpy
scikit-learn
joblib
prometheus-client
waitress
gunicorn
requests
"""
(H14 / "requirements.txt").write_text(reqs, encoding="utf-8")
print("[deps] Wrote:", H14 / "requirements.txt")

# ----------------------------------------------------------------------
# Dockerfile + start scripts
# ----------------------------------------------------------------------
dockerfile = '''FROM python:3.11-slim
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8000
# gunicorn app entrypoint (Docker/Linux)
CMD ["gunicorn","-w","2","-b","0.0.0.0:8000","src.app:app"]
'''
(H14 / "Dockerfile").write_text(dockerfile, encoding="utf-8")

(H14 / "start.ps1").write_text('pip install -r requirements.txt\r\npython src\\app.py\r\n', encoding="utf-8")
(H14 / "start.sh").write_text('#!/usr/bin/env bash\npip install -r requirements.txt\npython src/app.py\n', encoding="utf-8")


monitor_py = '''import time, csv, requests, os
from datetime import datetime

BASE = os.environ.get("API_BASE","http://127.0.0.1:8000")
OUT  = os.environ.get("MONITOR_LOG","docs/monitor_log.csv")
INTERVAL = float(os.environ.get("MONITOR_INTERVAL","10"))

print(f"[monitor] polling {BASE}/health every {INTERVAL}s -> {OUT}")
with open(OUT, "a", newline="") as f:
    w = csv.writer(f)
    if f.tell() == 0:
        w.writerow(["ts","ok","latency_ms"])
    while True:
        t0 = time.perf_counter()
        ok = False
        try:
            r = requests.get(f"{BASE}/health", timeout=5)
            ok = (r.status_code == 200)
        except Exception:
            ok = False
        dt_ms = int((time.perf_counter()-t0)*1000)
        w.writerow([datetime.utcnow().isoformat()+"Z", int(ok), dt_ms]); f.flush()
        time.sleep(INTERVAL)
'''
(SRC / "monitor.py").write_text(monitor_py, encoding="utf-8")

[model] Ready at: C:\Users\User\bootcamp_Khushi_Khanna\homework\homework14\model\model.pkl
[api] Wrote: C:\Users\User\bootcamp_Khushi_Khanna\homework\homework14\src\app.py
[deps] Wrote: C:\Users\User\bootcamp_Khushi_Khanna\homework\homework14\requirements.txt


826